In [2]:
import numpy as np
from numba import njit
import pandas as pd
import math
import heapq
from itertools import combinations

In [11]:
a = pd.read_csv("./a280.csv")
# a = pd.read_csv("./datasets/minitsp.csv")
points = np.array(a[['x','y']])
inputsize = len(points)

In [12]:
# distance matrix 만들기
diff = points[:, np.newaxis, :] - points[np.newaxis, :, :]
distance_matrix = np.sqrt(np.sum(diff ** 2, axis=-1))
len(distance_matrix)

280

In [5]:
# distance_matrix = [
#     [0, 2, 9, 10],
#     [1, 0, 6, 4],
#     [15, 7, 0, 8],
#     [6, 3, 12, 0],
# ]

S = distance_matrix
v = 0

memo = np.full((1 << len(S), len(S)), np.inf) # 비트마스킹으로 10001 이면 두개 원소 있는거임

# def dp(S, v): #v 는 인덱스 S 는 리스트
#     Min = np.inf
#     size = len( S )
#     if size == 1:
#         return distance_matrix[0][v]
#     for i in S:
#         if i == v:
#             continue
#         new_s = S.copy()
#         new_s.remove(v)
#         Min = min (Min, dp( new_s, i ) + distance_matrix[i][v] )
#     return Min

# def dp(S, v=0): #v 는 인덱스 S 는 리스트
#     Min = np.inf
#     size = len( S )
#     minindex=0
#     if size == 1:
#         memo[0][v]=distance_matrix[0][v]
#         return distance_matrix[0][v]
#     Sbitmask = 0
#     for x in S:
#         Sbitmask += 1<<x
#     if np.isfinite( memo[Sbitmask][v] ):
#         return memo[Sbitmask][v]
#     for i in S:
#         if i == v:
#             continue
#         new_s = S.copy()
#         new_s.remove(v)
#         cost = dp( new_s, i ) + distance_matrix[i][v]
#         if cost < Min:
#             Min = cost
#             minindex = i
#         memo[Sbitmask - (1<<minindex)][v] = Min
#     return Min

def dp(S, v=0): #v 는 인덱스 S 는 리스트
    Min = np.inf
    size = len( S )
    minindex=0
    if size == 1:
        memo[0][v]=distance_matrix[0][v]
        return distance_matrix[0][v]
    Sbitmask = 0
    for x in S:
        Sbitmask += 1<<x
    if np.isfinite( memo[Sbitmask][v] ):
        return memo[Sbitmask][v]
    for i in S:
        if i == v:
            continue
        new_s = S.copy()
        new_s.remove(v)
        cost = dp( new_s, i ) + distance_matrix[i][v]
        if cost < Min:
            Min = cost
            minindex = i
    memo[Sbitmask][v] = Min
    return Min


x = [x for x in range(len(S))]
b = np.array(x)
dp(x,0)
# memo
# 246.8
# np.delete(b,2)
# memo

238.8919583350927

In [6]:
def held_karp_iterative(dist):
    """
    Held-Karp 알고리즘의 반복 버전으로 TSP 문제를 해결
    
    Args:
        dist: 2차원 배열, dist[i][j]는 도시 i에서 j로 가는 비용
    
    Returns:
        (최소 비용, 최적 경로)
    """
    n = len(dist)
    N = 1 << n  # 2^n 개의 부분집합
    INF = float('inf')
    
    # dp[mask][j] = 부분집합 mask를 방문하고 도시 j에서 끝나는 최소 비용
    dp = [[INF] * n for _ in range(N)]
    
    # parent[mask][j] = (mask, j) 상태에서의 이전 도시
    parent = [[-1] * n for _ in range(N)]
    
    # 기저 사례: 시작점(0)에서 시작, mask = 1<<0
    dp[1][0] = 0
    
    # 모든 부분집합에 대해 반복 (시작점 0을 포함하는 것만)
    for mask in range(1, N):
        if not (mask & 1):  # 시작점 0이 포함되지 않으면 건너뛰기
            continue
            
        for j in range(1, n):  # 끝점 j (0이 아닌 도시들)
            if not (mask & (1 << j)):  # j가 현재 부분집합에 없으면 건너뛰기
                continue
                
            # j를 제외한 이전 부분집합
            prev_mask = mask ^ (1 << j)
            
            # 이전 도시들 중에서 최소 비용 찾기
            for k in range(n):
                if k == j:  # 같은 도시는 건너뛰기
                    continue
                if not (prev_mask & (1 << k)):  # k가 이전 부분집합에 없으면 건너뛰기
                    continue
                    
                cost = dp[prev_mask][k] + dist[k][j]
                if cost < dp[mask][j]:
                    dp[mask][j] = cost
                    parent[mask][j] = k
    
    # 모든 도시를 방문한 후 시작점으로 돌아가는 최소 비용 계산
    full_mask = (1 << n) - 1  # 모든 도시를 포함하는 마스크
    min_cost = INF
    last_city = -1
    
    for j in range(1, n):
        cost = dp[full_mask][j] + dist[j][0]
        if cost < min_cost:
            min_cost = cost
            last_city = j
    
    # 경로 복원
    path = []
    mask = full_mask
    current = last_city
    
    # 역순으로 경로 추적
    while current != -1:
        path.append(current)
        if mask == 1:  # 시작점만 남았으면 종료
            break
        next_city = parent[mask][current]
        mask ^= (1 << current)
        current = next_city
    
    path.reverse()
    path = [0] + path + [0]  # 시작점과 끝점 추가
    
    return min_cost, path

def create_distance_matrix(points):
    """좌표 배열로부터 거리 행렬 생성"""
    n = len(points)
    dist = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            if i != j:
                dx = points[i][0] - points[j][0]
                dy = points[i][1] - points[j][1]
                dist[i][j] = np.sqrt(dx*dx + dy*dy)
    
    return dist

In [ ]:
held_karp_iterative(distance_matrix)